# Preparing supply-use data for CVXLab

Prepare Italy's 2022 national supply-use table for the MARIO–CVXLab exercise.
Download the official ISTAT workbooks, attach compatible satellite accounts,
translate the labels into English, aggregate and inspect the database, then
export the reference to CVXLab. This tutorial ends at that handoff.

The exercise uses the published national classifications. Electricity, gas,
steam and air conditioning remain one activity; no electricity-technology
disaggregation or power-sector scenarios are introduced.

## Problem statement

Prepare a national supply-use reference with domestic uses, separate product
imports and compatible emissions, employment and energy accounts for a
CVXLab model.

Start from 63 ISTAT activities and products and obtain 15 of each, retaining
7 final-demand categories, 8 income/accounting rows and 27 satellite accounts.
Inspect the resulting reference and transfer it using the CVXLab model's
table names, coordinates and filters.

## Tutorial materials

Download this notebook, `aggregation.xlsx` and
`cvxlab_model.zip` from the [tutorial page](https://tutorials.marioproject.com/economy-wide-impact-assessment/index.html#tutorial-materials).
Extract the archive beside the notebook to obtain `cvxlab_model/`.
It contains the CVXLab `model_settings.xlsx`, classified sets
and reference input tables. The notebook regenerates
those inputs from the official source data.

The ISTAT parser additions are intended for MARIO 1.3.6 and are not in 1.3.5.
Install the development snapshot below in a dedicated Python 3.12
environment, then restart the notebook kernel after installation.

```sh
conda create -n mario_cvxlab_tutorial python=3.12 pip
conda activate mario_cvxlab_tutorial
python -m pip install "mariopy @ git+https://github.com/it-is-me-mario/MARIO.git@e786e43fd01186bf1707cf098e2ceea1afb29e49" "cvxlab==1.0.1b1" jupyterlab
python -m jupyter lab
```

The code uses relative paths from the working folder. The official monetary
release and the selected Eurostat snapshots are cached under `source_data/`.
The archive's `README.rst` records source attribution and the receiving-model
contract. Source data is downloaded separately rather than bundled here.

## Database download

Download the official [ISTAT 2020–2022 edition](https://www.istat.it/tavole-di-dati/il-sistema-di-tavole-input-output-anni-2020-2022/).
The parser selects 2022 at current basic prices. The native classification
contains 63 NACE Rev. 2 branches and 63 CPA 2.1 products, excluding
extraterritorial organisations. Existing cached workbooks are reused.

In [ ]:
import mario

mario.download_istat_io(path="source_data", edition="2020-2022")

## Database parsing

Read SUPPLY, USE and IMPORT and select the three optional satellite
groups. `download=True` reuses the monetary release and fetches missing
Eurostat snapshots for Italy in the same reference year.

ISTAT is already single-region: do not call `to_single_region()`.
`IMP_a` and `IMP_Y` retain imports by product and purchasing activity or final
demand. `U` and `Yc` contain domestic uses; no origin-country detail is available.

The selected accounts are [AEA emissions](https://ec.europa.eu/eurostat/databrowser/view/env_ac_ainah_r2/default/table?lang=en),
[employment](https://ec.europa.eu/eurostat/databrowser/view/nama_10_a64_e/default/table?lang=en)
and [PEFA net domestic energy use](https://ec.europa.eu/eurostat/databrowser/view/env_ac_pefa04/default/table?lang=en).
Hours worked come from the ISTAT USE workbook. Production accounts populate
`Ea`; direct household emissions and energy populate `EY`. `Ec` is zero.
Native NACE codes align the satellite observations before labels are translated.
Missing required observations cause an error rather than being filled with zero.

In [ ]:
db = mario.parse_istat(
    path="source_data",
    year=2022,
    table="SUT",
    level="63",
    price="current",
    valuation="basic",
    import_representation="separate",
    extensions=["emissions", "employment", "energy"],
    download=True,
    edition="2020-2022",
)

## Aggregation

`aggregation.xlsx` maps the original ISTAT labels directly to the English
classifications used by the model. Its Activity and Commodity sheets aggregate
63 branches and products into 15 groups. Consumption category, Factor of
production and Satellite account provide one-to-one English labels.

Apply all five sheets in one call. Imports follow the commodity aggregation;
final-demand categories, factors and satellite indicators remain separate.
The monetary unit is renamed `million EUR` without numerical rescaling.


In [ ]:
db.aggregate(
    io="aggregation.xlsx",
    levels=["Activity", "Commodity", "Consumption category",
            "Factor of production", "Satellite account"],
    calc_all=False,
    zero_output_epsilon=None,
)

for level in ["Activity", "Commodity", "Factor of production"]:
    db.units[level].loc[:, "unit"] = "million EUR"
db.meta.model_configuration["import_unit"] = "million EUR"

## Explore the reference

Inspect the classifications, units and representative blocks before
export. The combined SUT is 30 × 30, with a 30 × 1 production vector; it remains
a supply-use system rather than a symmetric IOT.

The 27 satellite rows comprise 24 AEA indicators, employed persons, hours worked
and net domestic energy use in TJ. `Greenhouse gases (CO2 equivalent) [AEA: GHG]`
is already characterised in t CO2-eq: do not sum it with overlapping gas
components or equivalent versions. These direct national accounts do not add
the emissions embodied in imported goods. Published monetary rounding
differences are retained.

In [ ]:
db.get_index("all")

In [ ]:
db.units["Satellite account"]

In [ ]:
print("Combined SUT:", db.query("Z").shape)
print("Production vector:", db.query("X").shape)
db.query("S").head()

In [ ]:
db.query("IMP_a").head()

In [ ]:
db.query("Ea").head()

## Export to CVXLab

`cvxlab_model/model_settings.xlsx` defines the CVXLab sets, input tables,
variables and equations. The `mapping` argument links MARIO classifications
and matrices to these sets and tables. Activities and final demand share the
`accounts` set; factors and satellites share `extensions`.

The export populates six reference tables: `make_0`, `use_0`, `imports_0`,
`extensions_0`, `total_activity_0` and `total_demand_0`. `use_0` combines
`U` and `Yc`; `imports_0` combines `IMP_a` and `IMP_Y`; `extensions_0`
combines `Va`, `Ea` and `EY`.

This configuration has no exogenous `VY` input. That final-demand accounting
block remains in the MARIO database. The `Reference` scenario also contains
blank `delta_make`, `delta_use` and `delta_imports` templates. CVXLab loads
them as zero according to `blank_fill=0`; they appear as unmapped tables in
the export report because they are scenario inputs rather than MARIO matrices.

`to_cvxlab` writes the classified sets to `sets.xlsx` and the input tables to
`input_data/input_data.xlsx` inside `cvxlab_model/`. Use that directory as
the model input for the CVXLab exercise.


In [ ]:
report = db.to_cvxlab(
    model_settings="cvxlab_model/model_settings.xlsx",
    path="cvxlab_model",
    mapping={
        "sets": {
            "Commodity_from": "products", "Commodity_to": "products",
            "Activity_from": "accounts", "Activity_to": "accounts",
            "Consumption category_to": "accounts",
            "Factor of production_from": "extensions",
            "Satellite account_from": "extensions",
        },
        "matrices": {
            "S": "make_0", "U": "use_0", "Yc": "use_0",
            "IMP_a": "imports_0", "IMP_Y": "imports_0",
            "Va": "extensions_0", "Ea": "extensions_0", "EY": "extensions_0",
            "Xa": "total_activity_0", "Xc": "total_demand_0",
        },
    },
)
report["written_cells"]